# BirdBase — Crosswalk Building & Feature Engineering

**Purpose:**  
Link every species in the *BirdBase* master dataset to a canonical **Avibase ID** using a multi-pass name-matching cascade across two external crosswalk references. Then produce two clean output tables:

| Output File | Description |
|-------------|-------------|
| `birdbase_crosswalk.csv` | Taxonomy lookup — Avibase ID + multi-taxonomy names per species |
| `birdbase_uncleaned_final_dataset.csv` | Trait dataset — selected ecological/functional features per species |

---

### Matching Strategy — 4-Pass Cascade

The key column `Latin (BirdLife > IOC > Clements>AviList)` contains species names that may follow different taxonomic conventions (BirdLife, IOC, eBird/Clements, AviList). We run four sequential passes against different reference name columns:

```
Pass 1 → avonet_crosswalk [ species_birdlife ]
Pass 2 → crosswalk        [ Birdlife_name    ]
Pass 3 → crosswalk        [ IOC_name         ]
Pass 4 → avonet_crosswalk [ species_ebird    ]
```

Each pass only processes species still **unmatched** from the previous one. Matched species are progressively accumulated into `birdbase_crosswalk`.

---
## 1 · Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

BASE = Path("../data/raw")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)
warnings.filterwarnings("ignore")

---
## 2 · Load Raw Data

In [5]:
crosswalk        = pd.read_csv(BASE / "crossWalk.csv", encoding="latin1")
avonet_crosswalk = pd.read_csv(BASE / "Avonet/avonet_crosswalk.csv", encoding="latin1")
birdbase         = pd.read_excel(BASE / "Birdbase/birdbase_raw.xlsx", sheet_name=0, skiprows=1)
# birdfunctiondata = pd.read_csv(BASE / "BirdFuncDat.txt", sep="\t", encoding="latin1")
# avian            = pd.read_csv(BASE / "avian_ssd_jan07.txt", sep="\t", encoding="latin1", on_bad_lines="warn")

print("Dataset shapes:")
for name, df in [("crosswalk", crosswalk), ("avonet_crosswalk", avonet_crosswalk), ("birdbase", birdbase)]:
    print(f"  {name:<20}: {df.shape}")

Dataset shapes:
  crosswalk           : (11167, 19)
  avonet_crosswalk    : (11237, 10)
  birdbase            : (11589, 97)


---
## 3 · Quick Exploration

In [ ]:
crosswalk.head()

In [ ]:
avonet_crosswalk.head()

In [ ]:
birdbase.head()

In [ ]:
# Key column diagnostics
KEY_COL = "Latin (BirdLife > IOC > Clements>AviList)"

print(f"BirdBase total rows           : {len(birdbase)}")
print(f"Unique species (key col)      : {birdbase[KEY_COL].nunique()}")
print(f"Null values in key col        : {birdbase[KEY_COL].isnull().sum()}")
print(f"avonet_crosswalk species_birdlife unique : {avonet_crosswalk['species_birdlife'].nunique()}")

---
## 4 · Preprocess — Standardise Avibase IDs in Crosswalk

The `crosswalk` table stores Avibase IDs in **lowercase** format (`avibase-XXXXXXXX`).  
All other datasets use **uppercase** (`AVIBASE-XXXXXXXX`).  
We normalise before matching to avoid false mismatches.

In [ ]:
# Before
print("Before:", crosswalk["TAXON_CONCEPT_ID"].iloc[0])

crosswalk["TAXON_CONCEPT_ID"] = crosswalk["TAXON_CONCEPT_ID"].str.upper()

# After
print("After :", crosswalk["TAXON_CONCEPT_ID"].iloc[0])
print(f"\nUnique IDs : {crosswalk['TAXON_CONCEPT_ID'].nunique()}")
print(f"Null IDs   : {crosswalk['TAXON_CONCEPT_ID'].isnull().sum()}")

---
## 5 · Build `birdbase_crosswalk` — Multi-Pass Species Matching

### 5.1 · Matching Function

`match_and_append` takes the current pool of unmatched species and a reference dataset, performs exact name matching (after whitespace stripping), prints a summary report, and returns:
- `new_matches` — `(avibase_id, latin_name)` for all newly matched species  
- `unmatched_df` — remaining species with no match yet

In [ ]:
def match_and_append(unmatched_df, ref_df, ref_col, id_col,
                     birdbase_col="Latin (BirdLife > IOC > Clements>AviList)"):
    """
    Match BirdBase species names against a reference name column.

    Parameters
    ----------
    unmatched_df : DataFrame  — species still needing an Avibase ID
    ref_df       : DataFrame  — reference dataset to match against
    ref_col      : str        — species name column in ref_df
    id_col       : str        — Avibase ID column in ref_df
    birdbase_col : str        — key name column in unmatched_df

    Returns
    -------
    new_matches  : DataFrame  — matched rows with (avibase_id, latin_name)
    unmatched_df : DataFrame  — species still unmatched after this pass
    """
    birdbase_names = unmatched_df[birdbase_col].dropna().str.strip()
    ref_names      = set(ref_df[ref_col].dropna().str.strip())

    matched   = birdbase_names[birdbase_names.isin(ref_names)]
    unmatched = birdbase_names[~birdbase_names.isin(ref_names)]

    total, n_matched, n_unmatched = len(birdbase_names), len(matched), len(unmatched)

    print(f"  Total       : {total}")
    print(f"  Matched     : {n_matched}  ({n_matched / total * 100:.1f}%)")
    print(f"  Not matched : {n_unmatched}  ({n_unmatched / total * 100:.1f}%)")

    new_matches = (
        ref_df[ref_df[ref_col].str.strip().isin(matched.values)]
        [[id_col, ref_col]]
        .rename(columns={id_col: "avibase_id", ref_col: "latin_name"})
        .drop_duplicates()
        .reset_index(drop=True)
    )

    unmatched_df = unmatched.reset_index(drop=True).to_frame()
    return new_matches, unmatched_df

### 5.2 · Run the 4-Pass Cascade

In [ ]:
# Pass 1 — avonet_crosswalk : species_birdlife
print("Pass 1 — avonet_crosswalk [species_birdlife]")
new_matches, unmatched_df = match_and_append(birdbase, avonet_crosswalk, "species_birdlife", "avibase_id")
birdbase_crosswalk = new_matches

# Pass 2 — crosswalk : Birdlife_name
print("\nPass 2 — crosswalk [Birdlife_name]")
new_matches, unmatched_df = match_and_append(unmatched_df, crosswalk, "Birdlife_name", "TAXON_CONCEPT_ID")
birdbase_crosswalk = pd.concat([birdbase_crosswalk, new_matches]).drop_duplicates().reset_index(drop=True)

# Pass 3 — crosswalk : IOC_name
print("\nPass 3 — crosswalk [IOC_name]")
new_matches, unmatched_df = match_and_append(unmatched_df, crosswalk, "IOC_name", "TAXON_CONCEPT_ID")
birdbase_crosswalk = pd.concat([birdbase_crosswalk, new_matches]).drop_duplicates().reset_index(drop=True)

# Pass 4 — avonet_crosswalk : species_ebird
print("\nPass 4 — avonet_crosswalk [species_ebird]")
new_matches, unmatched_df = match_and_append(unmatched_df, avonet_crosswalk, "species_ebird", "avibase_id")
birdbase_crosswalk = pd.concat([birdbase_crosswalk, new_matches]).drop_duplicates().reset_index(drop=True)

print(f"\n{'='*48}")
print(f"  birdbase_crosswalk total species : {len(birdbase_crosswalk)}")
print(f"  Still unmatched (to be dropped)  : {len(unmatched_df)}")
print(f"{'='*48}")

### 5.3 · Quality Checks

In [ ]:
print(f"Duplicate avibase_id : {birdbase_crosswalk['avibase_id'].duplicated().sum()}")
print(f"Duplicate latin_name : {birdbase_crosswalk['latin_name'].duplicated().sum()}")
print(f"Null values          :\n{birdbase_crosswalk.isnull().sum()}")

### 5.4 · Deduplicate on `latin_name`

A species matched across multiple passes can produce multiple rows (same name, different Avibase IDs from different sources). We keep one row per unique species name.

In [ ]:
birdbase_crosswalk = birdbase_crosswalk.drop_duplicates(subset="latin_name").reset_index(drop=True)

print(f"After dedup — rows            : {len(birdbase_crosswalk)}")
print(f"Remaining duplicate avibase_id: {birdbase_crosswalk['avibase_id'].duplicated().sum()}")

---
## 6 · Enrich Crosswalk with Multi-Taxonomy Names

Join all taxonomy name columns from BirdBase onto the crosswalk so it serves as a complete species lookup table.

In [ ]:
taxonomy_cols = [
    "Latin (BirdLife > IOC > Clements>AviList)",
    "English Name (BirdLife > IOC > Clements>AviList)",
    "HBW/BirdLife International (v9.1)",
    "IOC World Bird List (v15.1)",
    "eBird/Clements (V2024b)",
    "AviList v1 2025",
    "Order",
    "Family IOC 15.1",
    "Family Clements v2024b",
    "Family HBW/BirdLife v9.1 (2024)",
    "Family AviList v1 2025",
    "Genus"
]

birdbase_crosswalk = (
    birdbase_crosswalk
    .merge(
        birdbase[taxonomy_cols],
        left_on="latin_name",
        right_on="Latin (BirdLife > IOC > Clements>AviList)",
        how="left"
    )
    .drop(columns="Latin (BirdLife > IOC > Clements>AviList)")
)

print(f"Shape after enrichment : {birdbase_crosswalk.shape}")

### 6.1 · Rename Columns — Clean Consistent Schema

In [ ]:
birdbase_crosswalk = birdbase_crosswalk.rename(columns={
    "latin_name"                                       : "species_latin_key",
    "English Name (BirdLife > IOC > Clements>AviList)" : "species_english",
    "HBW/BirdLife International (v9.1)"                : "species_birdlife",
    "IOC World Bird List (v15.1)"                      : "species_ioc",
    "eBird/Clements (V2024b)"                          : "species_ebird",
    "AviList v1 2025"                                  : "species_avilist",
    "Order"                                            : "order",
    "Family IOC 15.1"                                  : "family_ioc",
    "Family Clements v2024b"                           : "family_clements",
    "Family HBW/BirdLife v9.1 (2024)"                  : "family_birdlife",
    "Family AviList v1 2025"                           : "family_avilist",
    "Genus"                                            : "genus"
})

print("Final columns:")
print(birdbase_crosswalk.columns.tolist())
birdbase_crosswalk.head()

---
## 7 · Clean BirdBase — Drop Unmatched Species & Add Avibase ID

### 7.1 · Drop Species with No Avibase ID Match

In [ ]:
before = len(birdbase)

birdbase = (
    birdbase[
        ~birdbase["Latin (BirdLife > IOC > Clements>AviList)"]
        .str.strip()
        .isin(unmatched_df["Latin (BirdLife > IOC > Clements>AviList)"])
    ]
    .reset_index(drop=True)
)

print(f"Rows before drop : {before}")
print(f"Rows after drop  : {len(birdbase)}")
print(f"Dropped          : {before - len(birdbase)}")

### 7.2 · Add Avibase ID to BirdBase

In [ ]:
birdbase = (
    birdbase
    .merge(
        birdbase_crosswalk[["species_latin_key", "avibase_id"]],
        left_on="Latin (BirdLife > IOC > Clements>AviList)",
        right_on="species_latin_key",
        how="left"
    )
    .drop(columns="species_latin_key")
)

print(f"Shape            : {birdbase.shape}")
print(f"Missing avibase_id : {birdbase['avibase_id'].isna().sum()}")

---
## 8 · Select & Rename Final BirdBase Features

Keep only relevant ecological and functional trait columns. Order/Family/Genus are retained in `birdbase_crosswalk` and can be joined on demand via `avibase_id`.

In [ ]:
birdbase = birdbase[[
    # identity
    "avibase_id",
    "Latin (BirdLife > IOC > Clements>AviList)",
    "Order",
    "Family HBW/BirdLife v9.1 (2024)",
    "Genus",
    # body mass
    "Female MinMass", "Female MaxMass",
    "Male MinMass",   "Male MaxMass",
    "Unsexed MinMass","Unsexed MaxMass",
    "Average Mass",
    # elevation
    "Xmin", "NormMin", "Elevational Range", "NormMax", "Xmax",
    # habitat
    "Primary Habitat",
    # diet
    "Primary Diet",
    "IN-Wt", "FR-Wt", "NE-Wt", "SE-Wt", "VE-Wt",
    "FI-Wt", "SC-Wt", "PL-Wt", "MS-Wt", "SUM-Wt",
    # reproductive
    "Nest_Type",
    "Flightlessness",
    # migration
    "Mig", "Alt", "Irreg", "Disp", "Sed"
]]

print(f"Selected features : {birdbase.shape[1]}  |  Rows : {birdbase.shape[0]}")

In [ ]:
birdbase = birdbase.rename(columns={
    "Latin (BirdLife > IOC > Clements>AviList)" : "species_latin",
    "Order"                                     : "order",
    "Family HBW/BirdLife v9.1 (2024)"           : "family",
    "Genus"                                     : "genus",
    # body mass
    "Female MinMass"                            : "mass_female_min",
    "Female MaxMass"                            : "mass_female_max",
    "Male MinMass"                              : "mass_male_min",
    "Male MaxMass"                              : "mass_male_max",
    "Unsexed MinMass"                           : "mass_unsexed_min",
    "Unsexed MaxMass"                           : "mass_unsexed_max",
    "Average Mass"                              : "mass_avg",
    # elevation
    "Xmin"                                      : "elev_min",
    "NormMin"                                   : "elev_norm_min",
    "Elevational Range"                         : "elev_range",
    "NormMax"                                   : "elev_norm_max",
    "Xmax"                                      : "elev_max",
    # habitat
    "Primary Habitat"                           : "habitat_primary",
    # diet
    "Primary Diet"                              : "diet_primary",
    "IN-Wt"                                     : "diet_invertebrate",
    "FR-Wt"                                     : "diet_fruit",
    "NE-Wt"                                     : "diet_nectar",
    "SE-Wt"                                     : "diet_seed",
    "VE-Wt"                                     : "diet_vertebrate",
    "FI-Wt"                                     : "diet_fish",
    "SC-Wt"                                     : "diet_scavenge",
    "PL-Wt"                                     : "diet_plant",
    "MS-Wt"                                     : "diet_mushroom",
    "SUM-Wt"                                    : "diet_sum",
    # reproductive
    "Nest_Type"                                 : "nest_type",
    "Flightlessness"                            : "flightless",
    # migration
    "Mig"                                       : "migratory",
    "Alt"                                       : "altitudinal_migrant",
    "Irreg"                                     : "irregular_migrant",
    "Disp"                                      : "dispersive",
    "Sed"                                       : "sedentary"
})

print("Final BirdBase columns:")
print(birdbase.columns.tolist())
birdbase.head()

---
## 9 · Final Validation

In [ ]:
print("── birdbase ──────────────────────────────────")
print(f"  Shape              : {birdbase.shape}")
print(f"  Missing avibase_id : {birdbase['avibase_id'].isna().sum()}")
print(f"  Duplicate avibase_id : {birdbase['avibase_id'].duplicated().sum()}")

print("\n── birdbase_crosswalk ────────────────────────")
print(f"  Shape              : {birdbase_crosswalk.shape}")
print(f"  Missing avibase_id : {birdbase_crosswalk['avibase_id'].isna().sum()}")
print(f"  Duplicate latin_name : {birdbase_crosswalk['species_latin_key'].duplicated().sum()}")

---
## 10 · Save Outputs

In [ ]:
birdbase.to_csv(
    "../data/raw/Birdbase/birdbase_uncleaned_final_dataset.csv",
    index=False,
    float_format="%.5f"
)
print("Saved : ../data/raw/Birdbase/birdbase_uncleaned_final_dataset.csv")

birdbase_crosswalk.to_csv(
    "../data/raw/Birdbase/birdbase_crosswalk.csv",
    index=False,
)
print("Saved : ../data/raw/Birdbase/birdbase_crosswalk.csv")